In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [3]:
# --- Parameters and quantum random-bit generator ---

# Number of qubits used in the protocol. Easy to change.
N_QUBITS = 100

# Fraction of matching-basis bits sacrificed as check bits to estimate errors.
CHECK_FRACTION = 0.5

# Detection threshold on the proportion of disrupted check bits.
# Intercept-resend gives ~25% errors, honest channel gives 0%, so 0.15 is safe.
ERROR_THRESHOLD = 0.15

# Whether Eve is active. Flip to False to confirm the detection
# logic does not raise a false alarm when no one is listening.
EVE_ACTIVE = True

# Simulator setup. Prefer Qiskit Aer when available (much faster and works
# well in Google Colab where `qiskit-aer` can be pip-installed). Fall back
# to Qiskit's built-in BasicSimulator if Aer is not installed.
try:
    from qiskit_aer import AerSimulator
    simulator = AerSimulator()
    SIMULATOR_NAME = "AerSimulator"
except Exception:
    from qiskit.providers.basic_provider import BasicSimulator
    simulator = BasicSimulator()
    SIMULATOR_NAME = "BasicSimulator"
print(f"Using simulator: {SIMULATOR_NAME}")

def quantum_random_bit():
    """Return a random bit 0/1 by measuring the state H|0> = 1/sqrt(2)(|0>+|1>).

    Quantum logic: H takes |0> to an equal superposition of |0> and |1>;
    measurement collapses to 0 or 1 with probability 1/2 each.
    """
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    result = simulator.run(transpile(qc, simulator), shots=1).result()
    counts = result.get_counts()
    return int(next(iter(counts)))

def quantum_random_bits(n):
    return [quantum_random_bit() for _ in range(n)]


Using simulator: AerSimulator


In [4]:
# --- ALICE ---

alice_bits  = quantum_random_bits(N_QUBITS)
alice_bases = quantum_random_bits(N_QUBITS)   # 0 = standard, 1 = diagonal

def alice_prepare(bit, basis):
    """Prepare a qubit encoding `bit` in the given `basis`.

    X flips |0> -> |1> to set the bit; H rotates standard -> diagonal basis.
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

alice_qubits = [alice_prepare(b, ba) for b, ba in zip(alice_bits, alice_bases)]

print("Alice bits :", alice_bits)
print("Alice bases:", alice_bases)


Alice bits : [1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1]
Alice bases: [1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1]


In [5]:
# --- EVE (intercept-resend) ---

def measure_in_basis(qc, basis):
    """Measure a circuit's qubit in the given basis. Returns the bit observed.

    Quantum logic: we apply H first if the basis is diagonal, so that
    |+>,|-> map back to |0>,|1>, then we measure in the computational basis.
    """
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    result = simulator.run(transpile(qc, simulator), shots=1).result()
    return int(next(iter(result.get_counts())))

def prepare_in_basis(bit, basis):
    """Prepare a fresh qubit encoding `bit` in the given `basis`.

    Same logic as Alice's preparation step.
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

if EVE_ACTIVE:
    eve_bases = quantum_random_bits(N_QUBITS)   # quantum-random basis choices
    eve_bits  = []                              # what Eve observed
    channel_qubits = []                              # what Bob actually gets

    for qc, ab in zip(alice_qubits, eve_bases):
        observed = measure_in_basis(qc, ab)          # intercept + measure
        eve_bits.append(observed)
        channel_qubits.append(prepare_in_basis(observed, ab))   # resend
    print("Eve bases:", eve_bases)
    print("Eve bits :", eve_bits)
else:
    # Honest channel — used to sanity-check that we don't get false positives.
    eve_bases = None
    eve_bits  = None
    channel_qubits = list(alice_qubits)
    print("(Eve is not active — honest channel.)")


Eve bases: [1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0]
Eve bits : [1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1]


In [6]:
# --- BOB ---

bob_bases = quantum_random_bits(N_QUBITS)
bob_bits  = [measure_in_basis(qc, b) for qc, b in zip(channel_qubits, bob_bases)]

print("Bob bases:", bob_bases)
print("Bob bits :", bob_bits)


Bob bases: [1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1]
Bob bits : [1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0]


In [7]:
# --- SIFTING ---

matching_positions = [i for i in range(N_QUBITS) if alice_bases[i] == bob_bases[i]]

alice_sifted   = [alice_bits[i]   for i in matching_positions]
bob_sifted = [bob_bits[i] for i in matching_positions]

print("Matching positions:", matching_positions)
print("Alice sifted     :", alice_sifted)
print("Bob sifted       :", bob_sifted)
print("Sifted length     :", len(alice_sifted))


Matching positions: [0, 1, 2, 7, 9, 10, 14, 16, 21, 24, 25, 27, 28, 32, 34, 35, 37, 40, 44, 46, 49, 51, 52, 54, 59, 60, 61, 64, 65, 67, 68, 69, 70, 74, 79, 80, 81, 82, 86, 87, 89, 91, 92, 93, 94, 96, 98, 99]
Alice sifted     : [1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1]
Bob sifted       : [1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0]
Sifted length     : 48


In [8]:
# --- CHECK BITS, ERROR RATE, AND DECISION ---

# Quantum-random selection: each sifted position is taken as a check bit
# whenever a quantum coin flip comes up 1.
check_mask = [quantum_random_bit() for _ in alice_sifted]

# Safety: if the coin happened to pick 0 check bits but we have sifted bits
# available, force the first one to be a check bit so we can compute an
# error rate (avoids 0/0 in the detection step).
if len(alice_sifted) > 0 and sum(check_mask) == 0:
    check_mask[0] = 1
    print("(Quantum coin picked 0 check bits — forcing one for the error estimate.)")

check_indices_in_sifted = [i for i, m in enumerate(check_mask) if m == 1]
key_indices_in_sifted   = [i for i, m in enumerate(check_mask) if m == 0]

alice_check   = [alice_sifted[i]   for i in check_indices_in_sifted]
bob_check = [bob_sifted[i] for i in check_indices_in_sifted]

# Count mismatches between Alice and Bob on the check bits.
if len(alice_check) > 0:
    mismatches = sum(1 for a, b in zip(alice_check, bob_check) if a != b)
    error_rate = mismatches / len(alice_check)
else:
    # Reachable only if the sifted key itself is empty (very small N_QUBITS).
    mismatches = 0
    error_rate = 0.0

attack_detected = error_rate > ERROR_THRESHOLD

# Final key: the matching-basis bits that were NOT used as check bits.
alice_final_key   = [alice_sifted[i]   for i in key_indices_in_sifted]
bob_final_key = [bob_sifted[i] for i in key_indices_in_sifted]

print("Check positions  :", check_indices_in_sifted)
print("Check bits used  :", len(alice_check),
      "of", len(alice_sifted), "sifted bits")
print("Alice check bits :", alice_check)
print("Bob check bits   :", bob_check)
print("Mismatches       :", mismatches, "/", len(alice_check))
print(f"Error rate       : {error_rate:.3f}")
print(f"Threshold        : {ERROR_THRESHOLD:.3f}")
print()
if attack_detected:
    print(">>> ATTACK DETECTED — abort, do not use the key. <<<")
else:
    print(">>> No attack detected — keeping remaining matching bits as the final key. <<<")
    print("Alice final key  :", alice_final_key)
    print("Bob final key    :", bob_final_key)
    print("Final key length :", len(alice_final_key))
    print("Final keys match :", alice_final_key == bob_final_key)

# Note: with small N_QUBITS the number of check bits is small and the
# error-rate estimate is noisy, so detection results can vary run-to-run.
# Increase N_QUBITS for a more reliable estimate.


Check positions  : [2, 4, 5, 9, 12, 14, 19, 21, 22, 23, 26, 27, 28, 29, 35, 38, 39, 45]
Check bits used  : 18 of 48 sifted bits
Alice check bits : [0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1]
Bob check bits   : [0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1]
Mismatches       : 3 / 18
Error rate       : 0.167
Threshold        : 0.150

>>> ATTACK DETECTED — abort, do not use the key. <<<


In [ ]:
# --- RESULTS & ANALYSIS: Monte-Carlo study of detection performance ---
#
# Re-run the full protocol many times with Eve ON and many times with Eve OFF.
# Compare the resulting error-rate distributions and detection rates.
# All randomness is still quantum (uses quantum_random_bit()).

import statistics
import matplotlib.pyplot as plt

def run_attacker_once(n, eve_on):
    """One complete BB84 round with optional intercept-resend Eve.

    Returns a dict {error_rate, n_check, detected} or None if the run produced
    no sifted bits (only possible for very small n).
    """
    a_bits  = quantum_random_bits(n)
    a_bases = quantum_random_bits(n)
    sent    = [alice_prepare(b, ba) for b, ba in zip(a_bits, a_bases)]

    if eve_on:
        e_bases = quantum_random_bits(n)
        relayed = []
        for q, eb in zip(sent, e_bases):
            obs = measure_in_basis(q, eb)
            relayed.append(prepare_in_basis(obs, eb))
    else:
        relayed = sent

    b_bases = quantum_random_bits(n)
    b_bits  = [measure_in_basis(q, b) for q, b in zip(relayed, b_bases)]

    matches = [i for i in range(n) if a_bases[i] == b_bases[i]]
    if not matches:
        return None
    a_sift = [a_bits[i] for i in matches]
    b_sift = [b_bits[i] for i in matches]

    mask = [quantum_random_bit() for _ in a_sift]
    if sum(mask) == 0:
        mask[0] = 1   # force at least one check bit
    chk = [i for i, m in enumerate(mask) if m == 1]
    a_chk = [a_sift[i] for i in chk]
    b_chk = [b_sift[i] for i in chk]
    mism  = sum(1 for x, y in zip(a_chk, b_chk) if x != y)
    err   = mism / len(chk)
    return {"error_rate": err, "n_check": len(chk), "detected": err > ERROR_THRESHOLD}

# Monte-Carlo parameters. Each trial runs ~ 6*N_TRIAL single-qubit circuits with
# Eve, ~ 4*N_TRIAL without. Aer makes this much faster; sizes chosen to give
# a tight detection-rate estimate without dragging on BasicSimulator.
N_TRIAL = 80
N_RUNS  = 20

print(f"Running {N_RUNS} trials with Eve ON, then {N_RUNS} with Eve OFF "
      f"(this may take a couple of minutes)...")

on_results  = [r for r in (run_attacker_once(N_TRIAL, True)  for _ in range(N_RUNS)) if r]
off_results = [r for r in (run_attacker_once(N_TRIAL, False) for _ in range(N_RUNS)) if r]

def summarise(label, results):
    rates    = [r["error_rate"] for r in results]
    detected = [r["detected"]   for r in results]
    return {
        "label":     label,
        "n":         len(results),
        "mean":      statistics.mean(rates),
        "stdev":     statistics.stdev(rates) if len(rates) > 1 else 0.0,
        "min":       min(rates),
        "max":       max(rates),
        "det_count": sum(detected),
        "det_rate":  sum(detected) / len(results),
    }

on_s  = summarise("Eve ACTIVE  (intercept-resend)", on_results)
off_s = summarise("Eve INACTIVE (honest channel)",  off_results)

print()
print(f"=== Detection-performance comparison ({N_TRIAL} qubits/trial) ===")
print(f"{'Metric':<26}{'Eve ON':>14}{'Eve OFF':>14}{'Theory(on/off)':>18}")
print("-" * 72)
print(f"{'Trials run':<26}{on_s['n']:>14}{off_s['n']:>14}{'—':>18}")
print(f"{'Mean error rate':<26}{on_s['mean']:>14.3f}{off_s['mean']:>14.3f}"
      f"{'0.250 / 0.000':>18}")
print(f"{'Stdev error rate':<26}{on_s['stdev']:>14.3f}{off_s['stdev']:>14.3f}{'—':>18}")
print(f"{'Min error rate':<26}{on_s['min']:>14.3f}{off_s['min']:>14.3f}{'—':>18}")
print(f"{'Max error rate':<26}{on_s['max']:>14.3f}{off_s['max']:>14.3f}{'—':>18}")
print(f"{'Flagged as attack':<26}{on_s['det_count']:>10}/{on_s['n']:<3}"
      f"{off_s['det_count']:>10}/{off_s['n']:<3}{'~100% / 0%':>18}")
print(f"{'Detection rate':<26}{100*on_s['det_rate']:>13.1f}%{100*off_s['det_rate']:>13.1f}%"
      f"{'~100% / 0%':>18}")
print(f"\nThreshold used: {ERROR_THRESHOLD:.3f}")

# Histogram comparing the two error-rate distributions.
plt.figure(figsize=(8, 4))
bins = [i * 0.05 for i in range(13)]   # 0.00 to 0.60 in steps of 0.05
plt.hist([r["error_rate"] for r in off_results], bins=bins,
         alpha=0.7, label=f"Eve OFF (n={off_s['n']})", edgecolor="black")
plt.hist([r["error_rate"] for r in on_results],  bins=bins,
         alpha=0.7, label=f"Eve ON  (n={on_s['n']})",  edgecolor="black")
plt.axvline(ERROR_THRESHOLD, color="red", linestyle="--",
            label=f"threshold = {ERROR_THRESHOLD}")
plt.axvline(0.25, color="green", linestyle=":",
            label="theory mean with Eve = 0.25")
plt.xlabel("Observed error rate on check bits")
plt.ylabel("Number of trials")
plt.title(f"Error-rate distributions ({N_RUNS} trials × {N_TRIAL} qubits each)")
plt.legend()
plt.tight_layout()
plt.show()


Running 20 trials with Eve ON, then 20 with Eve OFF (this may take a couple of minutes)...
